<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/05_workflow_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 05 — Workflow Agents

> **⚡ Quick path** — this is one of the five modules of the ~2½-hour course preview (M01 → M02 → M05 → M06 → M07).

> **Where you are** — one agent, many tools so far.
> - **You can already:** build a single `LlmAgent` with tools, run it, read its events.
> - **New in this module:** agents that *contain* agents — `SequentialAgent`, `ParallelAgent`,
>   `LoopAgent` — and the `{key}` templating that connects them through state.
> - **No new Python** — composition is configuration.

Every demo so far had **one agent** doing all the work. Real tasks rarely fit that: you summarize *then* translate; you look three things up *at once*; you draft *and revise until it's good*. Notice the words in italics — *then*, *at once*, *until*. Those are control-flow words, and ADK gives you one class per word:

- **`SequentialAgent`** — run children in order. Like a shell pipeline.
- **`ParallelAgent`** — run children at the same time. Like three browser tabs loading at once.
- **`LoopAgent`** — run children in a cycle until one of them says stop.

**What we'll do:**
1. A two-step pipeline: a summarizer feeds a translator.
2. A three-way fan-out: three researchers working at the same time.
3. The showcase: a generator + critic loop that refines a tagline until the critic stops complaining.
4. Nest them — and learn when *not* to use them.

**Running cost:** under $0.01 across all demos.

# Setup

Same ritual as every module: install, key, imports.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## API Key

In [2]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


## Imports

New here: the three workflow classes — `SequentialAgent`, `ParallelAgent`, `LoopAgent` — and the `exit_loop` tool our critic will use to stop a loop.

In [3]:
import os
import sys, warnings, asyncio, time, uuid, logging
warnings.filterwarnings("ignore")
try:
    sys.stderr.fileno()
except Exception:
    sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import exit_loop
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# The Three Shapes — One Picture

```
SequentialAgent           ParallelAgent                  LoopAgent
─────────────────         ─────────────────              ─────────────────
                                                         ┌────────────────┐
    child 1                     ┌─> child 1 ─┐           │   child 1      │
       │                        │            │           │      │         │
       ▼                  start ┼─> child 2 ─┼─> done    │      ▼         │
    child 2                     │            │           │   child 2      │
       │                        └─> child 3 ─┘           │      │         │
       ▼                                                 │      ▼ (loop)  │
    child 3                                              │   child 1 ...  │
                                                         └────────────────┘
    one after the         all three at once;             until exit_loop
    other, in order       wait for the slowest           or max_iterations
```

Same `Runner`, same event stream, same session state — only the wrapper class changes. (Some frameworks make you draw this as a node-and-edge graph; ADK just names the three shapes.)

What connects the children is **state**: one child writes its result there, the next one reads it. How exactly — in the first demo.

# SequentialAgent — Your First Pipeline

The task: take a rambling text, summarize it, then translate the summary into Slovak. Two jobs, fixed order — the translator has to wait for the summarizer.

Two new arguments make the hand-over work. On the summarizer:

```python
output_key="summary",
```

— "whatever this agent answers, also save it into `state["summary"]`." And in the translator's instruction:

```python
instruction="Translate this English sentence to Slovak: {summary}. ..."
```

— that `{summary}` is a placeholder ADK fills in **from session state** right before the model sees the prompt. Then the wrapper itself:

```python
pipeline = SequentialAgent(name="summarize_then_translate", sub_agents=[summarizer, translator])
```

A `SequentialAgent` has no model and no instruction of its own — it is pure order: run `sub_agents` left to right. The `run()` helper below is our `chat()` from earlier modules with one addition: it returns the session state at the end, so we can see what the agents wrote there.

In [4]:
APP = "m05_demos"
USER = "student"
MODEL = LiteLlm(model=MODEL_STRING)
ss = InMemorySessionService()

summarizer = LlmAgent(
    name="summarizer",
    model=MODEL,
    description="Summarizes the user input in one sentence.",
    instruction="Read the user input. Summarize it in one clear sentence. Output only the summary.",
    output_key="summary",
)

translator = LlmAgent(
    name="translator",
    model=MODEL,
    description="Translates English to Slovak.",
    # {summary} is auto-substituted from session state before the model sees the prompt.
    instruction="Translate this English sentence to Slovak: {summary}. Output only the translation.",
    output_key="translation",
)

pipeline = SequentialAgent(
    name="summarize_then_translate",
    sub_agents=[summarizer, translator],
)

async def run(agent, prompt: str, session_id_prefix: str):
    sid = f"{session_id_prefix}-{uuid.uuid4().hex[:6]}"
    await ss.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER: {prompt}\n")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.thought:  # the model's private reasoning summary (you saw it in M02) — skip it here
                    continue
                if p.text and p.text.strip():
                    print(f"[{ev.author}] {p.text.strip()[:250]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"[tool_call] {p.function_call.name}({args})")
    s = await ss.get_session(app_name=APP, user_id=USER, session_id=sid)
    return dict(s.state)

state = await run(pipeline, "The cat sat on the mat. It was hungry. It meowed loudly until fed.", "seq")
print("\n── Final state ──")
for k, v in state.items():
    print(f"  {k!r}: {v!r}")

USER: The cat sat on the mat. It was hungry. It meowed loudly until fed.



[summarizer] A hungry cat sat on a mat and meowed loudly until it was fed.


[translator] Hladná mačka sedela na podložke a hlasno mňaukala, kým ju nenakŕmili.

── Final state ──
  'summary': 'A hungry cat sat on a mat and meowed loudly until it was fed.'
  'translation': 'Hladná mačka sedela na podložke a hlasno mňaukala, kým ju nenakŕmili.'


### 🔍 What just happened?

- The summarizer answered first — and thanks to `output_key="summary"`, its answer also landed in state.
- The translator's `{summary}` was replaced with that value before its model call — it translated the summary, not your original text.
- The final state holds both keys. The pipeline left a paper trail you can inspect.

### ⚠️ Those braces are not Python's doing

`"Translate this English sentence to Slovak: {summary}"` is an ordinary string. To Python, `{summary}` is just nine characters — nothing gets filled in. (If you know f-strings: notice there is no `f` before the quote. If you don't: nothing to unlearn.)

The filling-in is **ADK's** job. Right before the model sees an instruction, ADK looks for `{...}` in it and swaps in the value stored under that key in session state. Write `{summary}` for a required key (ADK raises an error if it's missing) and `{summary?}` for an optional one — we'll need the `?` version in the loop demo.

### 🎯 Mini-task

Add a third step after the translator: an agent that writes a haiku about the translation (`output_key="haiku"`). Run the pipeline — do all three keys appear in the final state?

# ParallelAgent — Three at Once

Next task: three independent lookups — one fun fact each about Germany, Slovakia and Czechia. None of them needs the others' results, so why make them wait in line?

```python
trio = ParallelAgent(name="trio", sub_agents=[de_researcher, sk_researcher, cz_researcher])
```

Same shape as `SequentialAgent` — a wrapper with `sub_agents` and no model of its own — except it starts all children **at the same time**. Each child writes to its *own* `output_key` (`germany_fact`, `slovakia_fact`, `czech_fact`), so they can't overwrite each other. The cell also measures the total wall time — watch it.

One honest note before you run it: on its own, this trio is a research desk with no editor. Three facts land in state and nobody reads them. That is fine for a demo of *speed*; the useful version — someone on top who combines the three — is two demos down, in "Nesting Workflows".

In [5]:
de_researcher = LlmAgent(
    name="de_researcher", model=MODEL,
    instruction="Reply with exactly one sentence stating one interesting fun fact about Germany. No lists, no preamble.",
    output_key="germany_fact",
)
sk_researcher = LlmAgent(
    name="sk_researcher", model=MODEL,
    instruction="Reply with exactly one sentence stating one interesting fun fact about Slovakia. No lists, no preamble.",
    output_key="slovakia_fact",
)
cz_researcher = LlmAgent(
    name="cz_researcher", model=MODEL,
    instruction="Reply with exactly one sentence stating one interesting fun fact about the Czech Republic. No lists, no preamble.",
    output_key="czech_fact",
)

trio = ParallelAgent(
    name="trio",
    sub_agents=[de_researcher, sk_researcher, cz_researcher],
)

# Time the parallel run
t0 = time.time()
state = await run(trio, "Give me three country facts.", "par")
print(f"\n⏱  Total wall time: {time.time() - t0:.2f}s")
print("\n── Final state ──")
for k, v in state.items():
    if k.endswith("_fact"):
        print(f"  {k!r}: {v}")

USER: Give me three country facts.



[sk_researcher] Slovakia has the world’s highest number of castles and chateaux per capita.
[de_researcher] Germany is home to the world’s oldest continuously operating brewery, Weihenstephan, founded in 1040.


[cz_researcher] The Czech Republic has the world’s highest beer consumption per capita.

⏱  Total wall time: 1.81s

── Final state ──
  'slovakia_fact': Slovakia has the world’s highest number of castles and chateaux per capita.
  'germany_fact': Germany is home to the world’s oldest continuously operating brewery, Weihenstephan, founded in 1040.
  'czech_fact': The Czech Republic has the world’s highest beer consumption per capita.


### 🔍 What just happened?

- The authors interleave in whatever order they *finished*, not the order you declared them. That's what "at the same time" looks like in an event stream.
- The wall time is roughly **one** LLM call, not three. Three sequential ~1s calls would take ~3s; the fan-out finished in about a third of that — and the saving grows with every extra child.

### 🎯 Mini-task

Add a fourth researcher for Austria (`output_key="austria_fact"`) to the trio. Does the wall time change noticeably?

# LoopAgent — Generator and Critic

First drafts are never good. Human writers fix this with an editor: draft → feedback → revise → feedback… until the editor says "ship it". The loop below is exactly that, with two agents:

- a **generator** that writes (and later revises) a tagline, saving it to `state["draft"]`,
- a **critic** that checks three rules (specific, no marketing clichés, at most 12 words) and either writes a complaint into `state["critique"]` — or, once satisfied, calls the **`exit_loop`** tool, which ends the loop.

How do the two agents talk to each other? They don't — not directly. Nothing is passed as an argument, no object travels between them. **Session state is a whiteboard both of them can see.** The generator's `output_key="draft"` writes its answer on it; the critic's instruction contains `{draft?}`, so ADK copies the draft in before the critic's model call; the critic's `output_key="critique"` writes the complaint next to it; and in the next round the generator's `{critique?}` reads that. The loop itself only re-runs the children in order — the whiteboard carries over from round to round.

```
generator ── output_key="draft" ──>  state["draft"]     ── {draft?} ──>  critic
generator <── {critique?} ────────  state["critique"]  <── output_key="critique" ── critic
```

The wrapper:

```python
refiner = LoopAgent(name="tagline_refiner", sub_agents=[generator, critic], max_iterations=5)
```

— run the children in a cycle (generator, critic, generator, critic…) until someone calls `exit_loop`, or 5 rounds pass. `max_iterations` is the safety brake: if the critic can never be satisfied, the loop still stops. Always set it.

One detail to spot in the instructions below: `{draft?}` and `{critique?}` carry the question mark — in round 1 those keys don't exist yet, and `?` tells ADK "that's fine, leave it empty".

In [6]:
generator = LlmAgent(
    name="generator",
    model=MODEL,
    description="Writes or revises a tagline.",
    instruction="""Write or revise a 1-2 sentence tagline for a course on Google's Agent Development Kit (ADK) — the Python framework for building LLM agents,
aimed at software engineers. Requirements: specific, concrete, avoids marketing cliches
("unlock", "unleash", "empower", "transform").

If a previous draft and critique exist, revise the draft to address the critique.
Otherwise, write a fresh first draft.

Previous draft: {draft?}
Previous critique: {critique?}

Output ONLY the new tagline. No preamble, no markdown.""",
    output_key="draft",
)

critic = LlmAgent(
    name="critic",
    model=MODEL,
    description="Critiques drafts; calls exit_loop when satisfied.",
    instruction="""You critique taglines. Read this draft:

Draft: {draft}

Evaluate against these criteria:
- Does it say something SPECIFIC about the course content?
- Does it avoid marketing cliches ("unlock", "unleash", "empower", "transform", "elevate", "master")?
- Is it at most 12 words long? Count the words.

If the draft FAILS any criterion, write a one-sentence critique explaining which criterion failed and how.

If the draft PASSES all three criteria, call the exit_loop tool. Do NOT output text in that case.""",
    output_key="critique",
    tools=[exit_loop],
)

refiner = LoopAgent(
    name="tagline_refiner",
    sub_agents=[generator, critic],
    max_iterations=5,
)

state = await run(refiner, "Write a tagline.", "loop")
print("\n── Final draft ──")
print(state.get("draft", "(none)"))

USER: Write a tagline.



[generator] Build LLM agents in Python with Google’s Agent Development Kit—define tools, workflows, state, and multi-agent coordination in code.


[critic] The draft is specific and avoids clichés, but it fails the 12-word limit at 18 words.


[generator] Build LLM agents in Python with Google ADK: tools, workflows, state, coordination.


[tool_call] exit_loop({})

── Final draft ──
Build LLM agents in Python with Google ADK: tools, workflows, state, coordination.


### 🔍 What just happened?

Read the stream as rounds:

1. **Round 1** — the generator wrote a fresh draft (both `?` keys were empty). The critic counted the words, found too many, and wrote that complaint into state.
2. **Round 2** — the generator saw `{critique}` and cut the draft down. The critic checked again and called `exit_loop` — no text this time, just the tool call.
3. **The end** — the loop stopped because of that call, not because the 5-round brake ran out.

Models vary from run to run: if your critic was happy in round 1 and called `exit_loop` straight away, that is a valid result too — re-run the cell to see a revision.

Without `LoopAgent` you would write this yourself:

```python
for i in range(5):
    state["draft"] = generate(state)
    state["critique"] = critique(state)
    if satisfied(state["critique"]):
        break
```

The ADK version is the same idea — plus an event for every step, state you can audit afterwards, and the stop signal as an explicit tool call instead of guessing from the critic's wording.

### 🎯 Mini-tasks

1. **A looser critic.** Remove the 12-word rule and run the loop again. Does the critic now call `exit_loop` in round 1? Then try the opposite — a 6-word limit — and see whether the loop converges or hits `max_iterations`.
2. **A loop with one child.** A `LoopAgent` with a single child is legal: `LoopAgent(name="persistent_summarizer", sub_agents=[summarizer], max_iterations=3)`. Try it — does the same child really run three times?

# Nesting Workflows

A workflow agent is itself an agent — so it can be a child of another workflow. This is the piece missing from the parallel demo: an agent *on top* of the trio that reads the three facts and does something with them. The combination you'll actually use in production: **fan out the independent lookups in parallel, then synthesize in order.**

```python
pipeline2 = SequentialAgent(name="research_pipeline", sub_agents=[trio, synthesizer])
```

Read it plainly: step 1 is the whole `trio` fan-out from the previous demo; step 2 is a synthesizer that reads all three `{..._fact}` keys from state and writes a short report.

In [7]:
# Research three countries in parallel, then synthesize.
synthesizer = LlmAgent(
    name="synthesizer",
    model=MODEL,
    description="Writes a short combined report from three facts.",
    instruction="""You have three country facts in session state. Write a short
3-sentence combined report that connects them thematically if possible.

Germany: {germany_fact}
Slovakia: {slovakia_fact}
Czech Republic: {czech_fact}

Output only the report.""",
    output_key="report",
)

pipeline2 = SequentialAgent(
    name="research_pipeline",
    sub_agents=[
        trio,           # ← the ParallelAgent from earlier
        synthesizer,
    ],
)

t0 = time.time()
state = await run(pipeline2, "Research and synthesize.", "compose")
print(f"\n⏱  Total wall time: {time.time() - t0:.2f}s")
print("\n── Final report ──")
print(state.get("report", ""))

USER: Research and synthesize.



[cz_researcher] The Czech Republic has the world’s highest beer consumption per capita, with residents traditionally drinking more beer per person than anywhere else.


[de_researcher] Germany has more than 20,000 castles, ranging from medieval fortresses to ornate palaces.


[sk_researcher] Slovakia is home to Spiš Castle, one of the largest castle complexes in Central Europe.


[synthesizer] Germany, Slovakia, and the Czech Republic each offer distinctive cultural highlights rooted in Central European history and tradition. Germany boasts more than 20,000 castles, while Slovakia is home to Spiš Castle, one of Central Europe’s largest cas

⏱  Total wall time: 7.33s

── Final report ──
Germany, Slovakia, and the Czech Republic each offer distinctive cultural highlights rooted in Central European history and tradition. Germany boasts more than 20,000 castles, while Slovakia is home to Spiš Castle, one of Central Europe’s largest castle complexes. The Czech Republic complements this heritage with the world’s highest per-capita beer consumption, reflecting its enduring beer culture.


### 🔍 What just happened?

Four LLM calls ran — three researchers plus the synthesizer — but the wall time was close to two calls' worth, because the researchers overlapped. Parallel-inside-sequential is how production agents are usually put together: gather everything independent at once, then combine.

### 🎯 Mini-task

Swap the synthesizer's instruction for one that writes a single newspaper headline from the three facts. One instruction change, same pipeline.

# Workflow Agent, or Let the LLM Decide?

There is a second way to coordinate agents: give one `LlmAgent` the children and let its model pick what to do.

```python
# LLM-driven: the orchestrator's LLM picks which tool/sub-agent to call next.
orchestrator = LlmAgent(
    name="orchestrator",
    sub_agents=[summarizer, translator, researcher_de, ...],
    instruction="Figure out what to do based on user input.",
)
```

Both styles work. They're not interchangeable:

| Use a **workflow agent** when... | Use **LLM-driven** when... |
|---|---|
| The control flow is fixed — always summarize, then translate | The control flow depends on user input |
| You need determinism (tests, evals) | You need flexibility |
| Latency matters — Parallel starts everything at once, no extra model call to decide | The model's judgment is the point |
| You want the flow auditable from a diagram | Conversations can go anywhere |

Rule of thumb: **if you can name the workflow, use a workflow agent. If you can't, let the LLM decide.** And if a composition of Sequential/Parallel/Loop gets unwieldy, that's a hint the flow shouldn't be hard-named after all.

# Key Takeaways

- **Three workflow classes:** `SequentialAgent` (ordered pipeline), `ParallelAgent` (concurrent fan-out), `LoopAgent` (cycle until `exit_loop` or `max_iterations`).
- **State is the pipe:** `output_key="foo"` writes `state["foo"]`; `{foo}` in a later agent's instruction reads it; `{foo?}` means "optional, don't error if missing".
- **Always set `max_iterations`** on a LoopAgent — the critic might be impossible to satisfy.
- **`exit_loop`** is the stop signal. Give the tool to whichever child owns the exit condition.
- **Workflows nest:** Parallel inside Sequential = "fan out research, then synthesize".
- **Rule:** if you can name the workflow, use a workflow agent. If you can't, let the LLM decide.

# Next up — M06: Multi-agent hierarchies

Workflow agents give you control flow by declaration. M06 gives you the other half — **LLM-driven routing** between agents: `sub_agents` for handing a conversation over, `AgentTool` for consulting a specialist (you met it briefly in the tools module). Same agents, two coordination patterns, different trade-offs.